# ML Olist — Prédiction de la satisfaction client

## Projet : Analyse des ventes et prédiction de la satisfaction client sur la plateforme e-commerce Olist

Ce notebook construit un premier modèle de Machine Learning pour prédire si un client sera satisfait ou non.

## Problème métier

On cherche à prédire la satisfaction client à partir des informations de commande, de livraison, de paiement, de produit et de client.

## Target choisie

La variable d'origine est `review_score`, une note de 1 à 5.

Nous créons une target binaire :

- `1` = client satisfait si `review_score >= 4`
- `0` = client non satisfait si `review_score <= 3`

Ce choix est adapté à une soutenance car il est simple, clair et orienté métier.

## Modèles testés

Nous comparons trois modèles :

1. **Logistic Regression** : baseline simple et interprétable.
2. **Random Forest** : modèle robuste, adapté aux données tabulaires.
3. **Gradient Boosting** : modèle souvent performant sur les projets e-commerce.

## 1. Importation des bibliothèques

In [ ]:
# Manipulation des données.
import pandas as pd

# Calculs numériques.
import numpy as np

# Graphiques.
import matplotlib.pyplot as plt
import seaborn as sns

# Séparation train/test.
from sklearn.model_selection import train_test_split

# Préparation des variables.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Modèles de classification.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Métriques d'évaluation.
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Sauvegarde du modèle.
import joblib
import os

## 2. Chargement des fichiers nettoyés

In [ ]:
# Les fichiers nettoyés doivent avoir été générés par Cleaning_Olist.ipynb.
PROCESSED_DIR = "data/processed"

customers = pd.read_csv(f"{PROCESSED_DIR}/customers_clean.csv")
orders = pd.read_csv(f"{PROCESSED_DIR}/orders_clean.csv")
order_items = pd.read_csv(f"{PROCESSED_DIR}/order_items_clean.csv")
order_payments = pd.read_csv(f"{PROCESSED_DIR}/order_payments_clean.csv")
order_reviews = pd.read_csv(f"{PROCESSED_DIR}/order_reviews_clean.csv")
products = pd.read_csv(f"{PROCESSED_DIR}/products_clean.csv")
sellers = pd.read_csv(f"{PROCESSED_DIR}/sellers_clean.csv")
translation = pd.read_csv(f"{PROCESSED_DIR}/translation_clean.csv")

## 3. Vérification rapide des dimensions

In [ ]:
print("customers :", customers.shape)
print("orders :", orders.shape)
print("order_items :", order_items.shape)
print("order_payments :", order_payments.shape)
print("order_reviews :", order_reviews.shape)
print("products :", products.shape)
print("sellers :", sellers.shape)
print("translation :", translation.shape)

## 4. Création des variables de livraison

Les délais de livraison sont très importants pour la satisfaction client.

Nous créons :

- `delivery_time_days` : durée réelle entre achat et livraison ;
- `estimated_delivery_time_days` : durée estimée entre achat et livraison prévue ;
- `delivery_delay_days` : retard ou avance par rapport à la date estimée ;
- `is_late` : 1 si la commande est en retard, 0 sinon.

In [ ]:
orders_ml = orders.copy()

# Conversion des colonnes de dates après relecture des CSV.
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders_ml[col] = pd.to_datetime(orders_ml[col], errors="coerce")

# Délai réel de livraison.
orders_ml["delivery_time_days"] = (
    orders_ml["order_delivered_customer_date"] - orders_ml["order_purchase_timestamp"]
).dt.days

# Délai estimé de livraison.
orders_ml["estimated_delivery_time_days"] = (
    orders_ml["order_estimated_delivery_date"] - orders_ml["order_purchase_timestamp"]
).dt.days

# Retard de livraison.
orders_ml["delivery_delay_days"] = (
    orders_ml["order_delivered_customer_date"] - orders_ml["order_estimated_delivery_date"]
).dt.days

# Indicateur de retard.
orders_ml["is_late"] = np.where(orders_ml["delivery_delay_days"] > 0, 1, 0)

orders_features = orders_ml[[
    "order_id",
    "customer_id",
    "order_status",
    "delivery_time_days",
    "estimated_delivery_time_days",
    "delivery_delay_days",
    "is_late"
]]

orders_features.head()

## 5. Agrégation des articles au niveau commande

`order_items` est au niveau article. Pour le Machine Learning, nous voulons une ligne par commande.

Nous créons donc :

- nombre d'articles ;
- prix total ;
- prix moyen ;
- frais de livraison totaux ;
- nombre de vendeurs distincts.

In [ ]:
items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        nb_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        avg_price=("price", "mean"),
        total_freight=("freight_value", "sum"),
        nb_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

items_agg.head()

## 6. Ajout des informations produit

Une commande peut contenir plusieurs produits.

Pour garder une approche simple et académique, on récupère les informations du premier produit de chaque commande.

In [ ]:
items_products = order_items.merge(
    products[[
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]],
    on="product_id",
    how="left"
)

main_product = (
    items_products
    .sort_values(["order_id", "order_item_id"])
    .groupby("order_id")
    .first()
    .reset_index()
)

main_product = main_product[[
    "order_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]]

main_product.head()

## 7. Agrégation des paiements

Une commande peut avoir plusieurs paiements.

Nous créons :

- montant total payé ;
- nombre de séquences de paiement ;
- nombre maximum de mensualités ;
- type de paiement principal.

In [ ]:
payments_agg = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        nb_payment_sequences=("payment_sequential", "max"),
        max_payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

payment_type_main = (
    order_payments
    .sort_values(["order_id", "payment_sequential"])
    .groupby("order_id")
    .first()
    .reset_index()[["order_id", "payment_type"]]
)

payments_features = payments_agg.merge(payment_type_main, on="order_id", how="left")

payments_features.head()

## 8. Création de la target satisfaction

La target est construite à partir de `review_score`.

Si une commande a plusieurs avis, on utilise la moyenne des notes.

In [ ]:
reviews_agg = (
    order_reviews
    .groupby("order_id")
    .agg(review_score=("review_score", "mean"))
    .reset_index()
)

reviews_agg["is_satisfied"] = np.where(reviews_agg["review_score"] >= 4, 1, 0)

reviews_agg.head()

## 9. Variables clients

In [ ]:
customers_features = customers[[
    "customer_id",
    "customer_state",
    "customer_city"
]]

customers_features.head()

## 10. Construction de la table ML finale

Nous construisons une table au niveau commande.

Chaque ligne représente une commande avec ses informations de livraison, paiement, produit, client et satisfaction.

In [ ]:
ml_data = orders_features.copy()

ml_data = ml_data.merge(customers_features, on="customer_id", how="left")
ml_data = ml_data.merge(items_agg, on="order_id", how="left")
ml_data = ml_data.merge(main_product, on="order_id", how="left")
ml_data = ml_data.merge(payments_features, on="order_id", how="left")

# Jointure inner avec reviews : on garde les commandes pour lesquelles on connaît la satisfaction.
ml_data = ml_data.merge(reviews_agg, on="order_id", how="inner")

print("Dimensions de la table ML :", ml_data.shape)
ml_data.head()

## 11. Vérification de la table ML

In [ ]:
ml_data.isna().sum()

In [ ]:
ml_data["is_satisfied"].value_counts(normalize=True)

In [ ]:
ml_data["is_satisfied"].value_counts().sort_index().plot(kind="bar", figsize=(6, 4))
plt.title("Répartition de la satisfaction client")
plt.xlabel("0 = non satisfait, 1 = satisfait")
plt.ylabel("Nombre de commandes")
plt.xticks(rotation=0)
plt.show()

## 12. Sélection des variables du modèle

Nous excluons :

- `order_id` : identifiant technique ;
- `customer_id` : identifiant technique ;
- `review_score` : variable utilisée pour créer la target.

Nous conservons des variables explicatives disponibles dans les données opérationnelles.

In [ ]:
numeric_features = [
    "delivery_time_days",
    "estimated_delivery_time_days",
    "delivery_delay_days",
    "is_late",
    "nb_items",
    "total_price",
    "avg_price",
    "total_freight",
    "nb_sellers",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "total_payment_value",
    "nb_payment_sequences",
    "max_payment_installments"
]

categorical_features = [
    "order_status",
    "customer_state",
    "customer_city",
    "product_category_name",
    "payment_type"
]

X = ml_data[numeric_features + categorical_features]
y = ml_data["is_satisfied"]

print("X :", X.shape)
print("y :", y.shape)

## 13. Séparation train/test

On garde 80 % des données pour entraîner les modèles et 20 % pour les tester.

`stratify=y` permet de conserver la même proportion de clients satisfaits et non satisfaits dans les deux ensembles.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)

## 14. Préparation automatique des variables

Les variables numériques et catégorielles ne se préparent pas de la même manière.

Pour les variables numériques :

- remplacement des valeurs manquantes par la médiane ;
- standardisation.

Pour les variables catégorielles :

- remplacement des valeurs manquantes par `unknown` ;
- encodage OneHotEncoder.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 15. Modèle 1 — Logistic Regression

La régression logistique est notre baseline.

Elle sert de point de comparaison simple.

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]
)

logistic_model.fit(X_train, y_train)

y_pred_log = logistic_model.predict(X_test)
y_proba_log = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print(classification_report(y_test, y_pred_log))
print("ROC AUC :", roc_auc_score(y_test, y_proba_log))

## 16. Modèle 2 — Random Forest

La Random Forest est adaptée aux données tabulaires et capte mieux les relations non linéaires.

In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)
y_proba_rf = random_forest_model.predict_proba(X_test)[:, 1]

print("Random Forest")
print(classification_report(y_test, y_pred_rf))
print("ROC AUC :", roc_auc_score(y_test, y_proba_rf))

## 17. Modèle 3 — Gradient Boosting

Le Gradient Boosting est souvent performant sur les données tabulaires.

Il construit plusieurs arbres successifs, chaque arbre corrigeant une partie des erreurs précédentes.

In [ ]:
gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(random_state=42))
    ]
)

gradient_boosting_model.fit(X_train, y_train)

y_pred_gb = gradient_boosting_model.predict(X_test)
y_proba_gb = gradient_boosting_model.predict_proba(X_test)[:, 1]

print("Gradient Boosting")
print(classification_report(y_test, y_pred_gb))
print("ROC AUC :", roc_auc_score(y_test, y_proba_gb))

## 18. Comparaison des modèles

In [ ]:
model_results = pd.DataFrame({
    "model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "roc_auc": [
        roc_auc_score(y_test, y_proba_log),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_gb)
    ]
})

model_results.sort_values("roc_auc", ascending=False)

In [ ]:
model_results.sort_values("roc_auc").plot(
    x="model",
    y="roc_auc",
    kind="barh",
    legend=False,
    figsize=(8, 4)
)

plt.title("Comparaison des modèles selon le ROC AUC")
plt.xlabel("ROC AUC")
plt.ylabel("Modèle")
plt.show()

## 19. Sélection du meilleur modèle

On choisit automatiquement le modèle avec le meilleur ROC AUC.

In [ ]:
best_model_name = model_results.sort_values("roc_auc", ascending=False).iloc[0]["model"]

if best_model_name == "Logistic Regression":
    best_model = logistic_model
    y_pred_best = y_pred_log
elif best_model_name == "Random Forest":
    best_model = random_forest_model
    y_pred_best = y_pred_rf
else:
    best_model = gradient_boosting_model
    y_pred_best = y_pred_gb

print("Meilleur modèle :", best_model_name)

## 20. Matrice de confusion du meilleur modèle

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Non satisfait", "Satisfait"],
    yticklabels=["Non satisfait", "Satisfait"]
)

plt.title(f"Matrice de confusion - {best_model_name}")
plt.xlabel("Prédiction")
plt.ylabel("Valeur réelle")
plt.show()

## 21. Sauvegarde du meilleur modèle

Le modèle est sauvegardé dans le dossier `models/` pour pouvoir être réutilisé plus tard.

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(best_model, "models/best_satisfaction_model.joblib")

print("Modèle sauvegardé dans models/best_satisfaction_model.joblib")

## 22. Export d'un fichier de prédictions

Ce fichier pourra être chargé dans Neon ou utilisé dans Power BI.

In [ ]:
best_proba = best_model.predict_proba(X_test)[:, 1]

predictions = X_test.copy()
predictions["true_satisfaction"] = y_test.values
predictions["predicted_satisfaction"] = y_pred_best
predictions["satisfaction_probability"] = best_proba

predictions.to_csv("data/processed/ml_predictions.csv", index=False)

predictions.head()

## 23. Conclusion

Ce notebook a permis de construire un premier modèle ML complet :

1. préparation d'une table au niveau commande ;
2. création de la target satisfaction ;
3. comparaison de trois modèles ;
4. sélection du meilleur modèle ;
5. sauvegarde du modèle ;
6. export des prédictions.

## Modèle recommandé

Pour la soutenance, le modèle recommandé dépendra des résultats obtenus.

En général :

- la Logistic Regression est utile comme baseline ;
- la Random Forest est robuste et facile à défendre ;
- le Gradient Boosting est souvent performant sur des données e-commerce tabulaires.

## Limites

Ce modèle ne traite pas encore :

- les commentaires clients en NLP ;
- l'historique complet des clients ;
- les performances historiques des vendeurs ;
- les commandes multi-catégories de manière avancée.

Ce notebook constitue donc une première version académique et solide du modèle ML.